# TwoStageCNN1D: fit a selected evidential model and evaluate it on held-out stations

A causal convolution encodes each contiguous daily station segment. Three correction heads adjust its daily CMAQ sources; their physical sum sets the predictive mean.

**Prerequisites.** First save the station splits and complete the **TwoStageCNN1D** tuning notebook with the same feature option. This notebook consumes its saved result. We train on `pm25_filled`, monitor validation NLL on observed `pm25`, freeze the model, and only then read the test CSV. For independent model selection, compare candidates on validation data; treat the test result as a final report.

The example assumes PM2.5 concentrations in µg m⁻³. Run from the educational project root or this notebook's directory, using the project’s Python 3.11 kernel.

### 1. Choose the saved experiment

`data/splits` contains your real train, validation and test CSVs. `runs` contains one directory per model and feature option. Change these paths or set the indicated environment variables. Use a fixed feature option and station before inspecting test outcomes.

In [ ]:
from pathlib import Path
import hashlib
import json
import os

# Static inline figures avoid a desktop GUI dependency in Jupyter.
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from pm25_edl.nn_data import Preprocessor, load_split, validate_disjoint_stations
from pm25_edl.nn_models import build_model
from pm25_edl.nn_training import fit_model, predict_frame, seed_everything
from pm25_edl.metrics import score_predictions, score_training_predictions
from pm25_edl.plots import plot_pit, plot_discard, plot_station
from pm25_edl.tuning import load_best

cwd = Path.cwd()
root = next((candidate for p in [cwd, *cwd.parents]
             for candidate in (p, p / "educational")
             if (candidate / "src" / "pm25_edl").is_dir()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from the main MILES-GUESS project or an educational notebook directory.")

MODEL_NAME = "TwoStageCNN1D"
FEATURE_OPTION = os.environ.get("PM25_FEATURE_OPTION", "at-bb-aod")
SPLITS = Path(os.environ.get("PM25_DATA_DIR", root / "data" / "splits"))
RUNS = Path(os.environ.get("PM25_RUN_DIR", root / "runs"))
run_dir = RUNS / MODEL_NAME / FEATURE_OPTION
fit_dir, eval_dir = run_dir / "fit", run_dir / "evaluation"
STATION = None  # Or a station code. None = alphabetically first with an observation.
DATE_START, DATE_END = None, None  # Optional ISO dates for the timeline only.
REFIT = True   # False: evaluate a previously saved fit from this selection.
torch.set_num_threads(min(4, torch.get_num_threads()))

### 2. Load the selected architecture and preprocessing

Use `at-bb` (three source predictors) or `at-bb-aod` (plus AOD). A strict total-only `baseline` cannot correct individual sources. WRF and static variables stay fixed. The saved scalers were fitted using training stations only. Loading the tuning record also checks that the training and validation files and the saved preprocessor match this experiment. No tuning is run here.

In [ ]:
best = load_best(run_dir, model_name=MODEL_NAME, feature_option=FEATURE_OPTION, split_dir=SPLITS)
prep = Preprocessor.load(run_dir / "preprocessor.json")
train_df, val_df = load_split(SPLITS, "train"), load_split(SPLITS, "val")
validate_disjoint_stations(train_df, val_df)
train = prep.transform(train_df, split="train")
val = prep.transform(val_df, split="val")
print("Model:", MODEL_NAME, "| Feature option:", FEATURE_OPTION)
print("Resolved architecture:", best["model_kwargs"])
print("Training settings:", best["fit_kwargs"])
print(f"Rows: {len(train_df):,} train; {len(val_df):,} validation "
      f"({np.isfinite(val_df.pm25).sum():,} observed).")

### 3. Fit with the selected settings, then freeze the weights

The validation NLL determines the epoch to restore. The trained weights, a copy of the preprocessor, and a fingerprint of the tuning record are saved together. Set `REFIT=False` above on a later evaluation to reuse exactly those weights.

In [ ]:
fit_dir.mkdir(parents=True, exist_ok=True)
def file_sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

selection_id = file_sha256(run_dir / "best.json")
weight_file = fit_dir / "model.pt"
fit_record = fit_dir / "fit.json"

seed_everything(best["fit_kwargs"]["seed"])  # Seed before initializing weights.
model = build_model(MODEL_NAME, **prep.model_kwargs, **best["model_kwargs"])
if REFIT:
    history = fit_model(model, train, val, **best["fit_kwargs"])
    torch.save(model.state_dict(), weight_file)
    prep.save(fit_dir / "preprocessor.json")
    fit_record.write_text(json.dumps({"selection_sha256": selection_id, "weights_sha256": file_sha256(weight_file),
                                     "scaler_sha256": file_sha256(fit_dir / "preprocessor.json"),
                                     "best_epoch": history.best_epoch,
                                     "best_val_nll": history.best_val_nll}, indent=2), encoding="utf-8")
    pd.DataFrame({"train_loss": history.train_loss, "val_nll": history.val_nll},
                 index=np.arange(1, len(history.train_loss) + 1)).rename_axis("epoch").to_csv(fit_dir / "history.csv")

record = json.loads(fit_record.read_text(encoding="utf-8"))
if (record.get("selection_sha256") != selection_id or
    record.get("weights_sha256") != file_sha256(weight_file) or
    record.get("scaler_sha256") != file_sha256(fit_dir / "preprocessor.json")):
    raise ValueError("Fitted files differ from the recorded fit or HPT selection. Refit to freeze a new model.")
prep = Preprocessor.load(fit_dir / "preprocessor.json")
model = build_model(MODEL_NAME, **prep.model_kwargs, **best["model_kwargs"])
model.load_state_dict(torch.load(weight_file, map_location="cpu", weights_only=True))
model.eval()
print("Restored epoch:", record["best_epoch"], "| Validation NLL:", record["best_val_nll"])

### 4. Score the training fit and held-out station-days

For the test set, score only finite observed `pm25`, in the original units; never substitute an imputed value. Missing outcomes can still have valid predictions and remain in the exported CSV. NLL uses the total Student-t density. The mean Winkler score measures interval width and penalizes misses at the same coverage used in tuning. For R² higher is better; for the other four scores lower is better.

The training score uses `pm25_filled`, including imputed labels, at the same frozen model in evaluation mode. It measures fit to the training targets, not generalization. We save the training predictions/scores separately from the observed-only test results.

In [ ]:
train_predictions = predict_frame(model, train, batch_size=best["fit_kwargs"]["batch_size"])
assert len(train_predictions) == len(train_df)
train_scores = score_training_predictions(train_predictions, interval_level=best["interval_level"])
assert train_scores["n_obs"] == np.isfinite(train_df["pm25_filled"]).sum()
eval_dir.mkdir(parents=True, exist_ok=True)
train_predictions.to_csv(eval_dir / "train_predictions.csv", index=False)
train_summary = pd.DataFrame([train_scores], index=[MODEL_NAME])
train_summary.to_csv(eval_dir / "train_scores.csv", index_label="model")

test_df = load_split(SPLITS, "test")
validate_disjoint_stations(train_df, val_df, test_df)
test = prep.transform(test_df, split="test")
predictions = predict_frame(model, test, batch_size=best["fit_kwargs"]["batch_size"])
assert len(predictions) == len(test_df)
eval_dir.mkdir(parents=True, exist_ok=True)
predictions.to_csv(eval_dir / "test_predictions.csv", index=False)
scores = score_predictions(predictions, interval_level=best["interval_level"])
assert scores["n_obs"] == np.isfinite(test_df["pm25"]).sum()
print(f"Scored {scores['n_obs']:,} observed rows of {len(predictions):,}; "
      f"interval coverage level = {best['interval_level']:.0%}.")
summary = pd.DataFrame([scores], index=[MODEL_NAME])
summary.to_csv(eval_dir / "test_scores.csv", index_label="model")
manifest = {**record, "model": MODEL_NAME, "feature_option": FEATURE_OPTION,
            "test_csv_sha256": file_sha256(SPLITS / "test.csv"),
            "interval_level": best["interval_level"], "n_obs": int(scores["n_obs"])}
(eval_dir / "evaluation.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
display(pd.concat({"Training (pm25_filled)": train_summary,
                   "Test (observed pm25)": summary}, names=["split"]))

**Source accounting.** These columns are corrected component estimates in µg m⁻³. They sum to the model’s predicted total by construction. Only the total is supervised: conservation alone does not establish that the individual source estimates are correct. The model estimates uncertainty for the total, not for each source.

In [ ]:
source_cols = ["s_corrected_cmaq_pm25_" + source for source in ("bb", "anthro", "othr")]
# Same three-term sum in physical units as the exported prediction.
np.testing.assert_array_equal(predictions[source_cols].to_numpy().sum(axis=1), predictions["pred"])
display(predictions[["stcode", "date", *source_cols, "pred", "pm25"]].head())

### 5. Check the uncertainty

The normal–inverse-gamma head returns $(\gamma,\nu,\alpha,\beta)$ with $\nu,\beta>0$ and $\alpha>1$. In physical units, the predictive mean is $\gamma$ and
\[
\sigma^2_{\rm al}=\frac{\beta}{\alpha-1},\qquad
\sigma^2_{\rm ep}=\frac{\beta}{\nu(\alpha-1)},\qquad
\sigma^2_{\rm total}=\sigma^2_{\rm al}+\sigma^2_{\rm ep}.
\]
The first component is the model's expected conditional noise variance; the second is its estimated variance of the latent mean.

**PIT.** At each finite observed concentration $y$, evaluate the Student-t CDF centered at $\gamma$, with $2\alpha$ degrees of freedom and **scale** $\sqrt{\beta(1+1/\nu)/\alpha}$. This scale differs from the predictive standard deviation. A roughly flat histogram is consistent with marginal calibration; it does not establish calibration at every station or validate the decomposition.

In [ ]:
fig = plot_pit(predictions)
fig.savefig(eval_dir / "test_pit.png", dpi=180)
plt.show()

**Discard fraction.** Sort observed station-days by total predictive SD. Remove the most uncertain first and calculate RMSE on what remains. A falling curve suggests that the model tends to assign higher uncertainty to larger errors. This evaluates ranking, not probability calibration.

In [ ]:
fig = plot_discard(predictions)
fig.savefig(eval_dir / "test_discard.png", dpi=180)
plt.show()

**Station timeline.** Plot against date, with separate panels for $\gamma\pm\sigma_{\mathrm{al}}$ and $\gamma\pm\sigma_{\mathrm{ep}}$. The bands show different components around the same prediction. Neither is a calibrated prediction interval, and their widths should not be added. Missing dates break the forecast line; missing measurements are not filled.

In [ ]:
fig = plot_station(predictions, station=STATION, start=DATE_START, end=DATE_END)
fig.savefig(eval_dir / "test_station.png", dpi=180)
plt.show()
print("Results:", eval_dir.resolve())

### 6. Verify the implementation on a small mock batch

This final check is independent of the scientific results. It verifies the forward pass, finite loss, observed-target masking and, for two-stage architectures, exact conservation of source mass.

In [ ]:
from pm25_edl.verification import check_model_contract

check_model_contract(MODEL_NAME)
print(f"{MODEL_NAME}: schema and model checks passed.")